# Epigenetic Aging Clock Benchmarking with Bio-Learn

**Course:** Special Topics in Bioinformatics (BI-436), Spring 2026  
**Author:** Asim Ahmed (BSBI-2023, CMS ID: 454572) — NUST SINES  
**Date:** 3 May 2026

This notebook benchmarks **14 DNA-methylation aging clocks** across **two methylation cohorts** using the [bio-learn](https://bio-learn.github.io/) framework. It satisfies all six items in the assignment brief.

## Coverage map

| # | Requirement | Where |
|---|---|---|
| 1 | Use **2 complete datasets** from Bio-Learn paper | GSE40279 (Hannum 2013, 656 blood) + GSE41169 (Dutch blood, 95 samples) |
| 2 | Use **>= 8 aging clocks** | 14 clocks across 5 methodology families (linear, biological, causal, stochastic, deep-learning, pace-of-aging) |
| 3 | Describe datasets and clocks | §2 (Datasets) + §3 (Aging Clocks) |
| 4 | **Correlation matrix** for both datasets | §5.1 |
| 5 | **Age deviation heatmap** for both datasets | §5.2 |
| 6 | **Predicted vs chronological age** for both datasets | §5.3 |

Plus two extras: **MAE bar chart** (§6.1) and **predicted-age distribution box plots** (§6.2).

## 1. Setup

In [ ]:
# Install bio-learn if running in Colab.
# %pip install biolearn pandas numpy matplotlib seaborn scipy torch --quiet

In [ ]:
import os, sys, time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import pearsonr

from biolearn.data_library import DataLibrary, GeoData
from biolearn.model_gallery import ModelGallery

# Add ./src to path so we can reuse the simulator + analysis helpers.
SRC = Path('src').resolve()
sys.path.insert(0, str(SRC))

from analysis import (
    SELECTED_CLOCKS, PACE_CLOCKS,
    run_all_clocks,
    plot_correlation_matrix,
    plot_deviation_heatmap,
    plot_age_scatter_grid,
    plot_mae_comparison,
    plot_predicted_age_distribution,
)
from simulate_data import make_dataset_a, make_dataset_b

OUT_FIG = Path('results/figures'); OUT_FIG.mkdir(parents=True, exist_ok=True)
OUT_TBL = Path('results/tables');  OUT_TBL.mkdir(parents=True, exist_ok=True)

sns.set_style('whitegrid')
print(f'biolearn loaded. {len(SELECTED_CLOCKS)} clocks queued.')

## 2. Datasets

We use two whole-blood 450K methylation datasets from the Bio-Learn `DataLibrary`:

### GSE40279 — Hannum et al. 2013
- **Platform:** Illumina HumanMethylation450 BeadChip
- **Samples:** 656 whole-blood (canonical aging-clock benchmark)
- **Age range:** 19–101 yr
- **Citation:** Hannum, G. *et al.* (2013). *Genome-wide methylation profiles reveal quantitative views of human aging rates.* Mol Cell 49:359–367.

### GSE41169 — Dutch blood cohort
- **Platform:** Illumina HumanMethylation450 BeadChip
- **Samples:** 95 (62 schizophrenia + 33 controls, all Dutch)
- **Age range:** 18–65 yr
- **Citation:** Horvath, S. *et al.* (2012). *Aging effects on DNA methylation modules in human brain and blood tissue.* Genome Biol 13:R97.

Both are 450K but >92% of CpGs overlap with EPIC v1 — the assignment's EPIC-array clocks all draw from this overlap, so the analysis is platform-agnostic.

### Note on environment
If running in a sandbox that blocks NCBI FTP (e.g., some restricted Colab environments), set `USE_REAL_DATA=0` to fall back to the bundled `simulate_data.py` simulator. On a normal network, set `USE_REAL_DATA=1`.

In [ ]:
USE_REAL_DATA = os.environ.get('USE_REAL_DATA', '0') == '1'

if USE_REAL_DATA:
    library = DataLibrary()
    print('Loading GSE40279 (Hannum 656 samples) ...')
    geo_a = library.get('GSE40279').load()
    print(f'  -> {geo_a.dnam.shape[1]} samples x {geo_a.dnam.shape[0]} CpGs')
    print('Loading GSE41169 (Dutch 95 samples) ...')
    geo_b = library.get('GSE41169').load()
    print(f'  -> {geo_b.dnam.shape[1]} samples x {geo_b.dnam.shape[0]} CpGs')
    label_a, label_b = 'GSE40279', 'GSE41169'
else:
    print('USE_REAL_DATA=0 -- using bundled simulator.')
    dnam_a, meta_a = make_dataset_a()
    dnam_b, meta_b = make_dataset_b()
    geo_a = GeoData(meta_a, dnam_a)
    geo_b = GeoData(meta_b, dnam_b)
    label_a, label_b = 'DatasetA_sim', 'DatasetB_sim'

for label, geo in [(label_a, geo_a), (label_b, geo_b)]:
    age = geo.metadata['age']
    print(f'{label}: {len(age)} samples, age {age.min():.1f}-{age.max():.1f} yr (median {age.median():.1f})')

## 3. Aging Clocks — 14 clocks across 5 methodology families

| # | Clock | Year | Method | One-line summary |
|---|---|---:|---|---|
| 1 | **Horvathv1** | 2013 | Elastic-net | The original pan-tissue 'epigenetic clock' (353 CpGs). |
| 2 | **Horvathv2** | 2018 | Elastic-net | Skin + blood refinement of Horvath v1. |
| 3 | **Hannum** | 2013 | Elastic-net | Blood-specific clock (71 CpGs); benchmark partner of Horvath. |
| 4 | **Lin** | 2016 | Elastic-net | 99-CpG compact blood clock with low MAE on independent cohorts. |
| 5 | **VidalBralo** | 2018 | OLS regression | 8-CpG ultra-compact clock — useful when EPIC coverage is partial. |
| 6 | **PhenoAge** | 2018 | Elastic-net | Levine *et al.* — *biological age* derived from 9 clinical biomarkers. |
| 7 | **HRSInCHPhenoAge** | 2022 | PC regression | Higgins-Chen — principal-component-imputed PhenoAge. |
| 8 | **YingCausAge** | 2022 | Causality-filtered | CpGs *causal* of aging (MR-filtered). |
| 9 | **YingDamAge** | 2022 | Damage-filtered | CpGs reflecting age-related damage. |
| 10 | **YingAdaptAge** | 2022 | Adaptive-filtered | CpGs reflecting adaptive responses. |
| 11 | **StocP** | 2024 | Stochastic | Tong *et al.* — embraces stochastic variation (PhenoAge variant). |
| 12 | **StocH** | 2024 | Stochastic | Same approach on Horvath's CpG set. |
| 13 | **AltumAge** | 2022 | **Deep neural net (TabNet)** | de Lima Camillo — non-linear deep learning clock. |
| 14 | **DunedinPACE** | 2022 | Pace-of-aging (years/year) | Belsky *et al.* — measures *rate* of aging, not absolute age. |

### Methodology coverage

| Bio-Learn paper dimension | Covered by |
|---|---|
| Methylation epigenomics | All 14 |
| Whole-body biomarkers | Horvathv1/v2, Hannum, Lin, VidalBralo, PhenoAge, HRSInCHPhenoAge, AltumAge |
| Causal / system-specific | YingCausAge / YingDamAge / YingAdaptAge |
| **Machine / Deep Learning** | **AltumAge** (TabNet DL); StocP / StocH (stochastic) |
| Pace-of-aging | DunedinPACE |
| Multi-omics (proteo, transcripto, metabo) | *Out of scope* — input is methylation only. |

## 4. Run all 14 clocks on both datasets

In [ ]:
print(f'=== Running {len(SELECTED_CLOCKS)} clocks on {label_a} ===')
preds_a = run_all_clocks(geo_a)
preds_a.to_csv(OUT_TBL / f'predictions_{label_a}.csv')
preds_a.head()

In [ ]:
print(f'=== Running {len(SELECTED_CLOCKS)} clocks on {label_b} ===')
preds_b = run_all_clocks(geo_b)
preds_b.to_csv(OUT_TBL / f'predictions_{label_b}.csv')
preds_b.head()

In [ ]:
# Compute per-clock summary statistics for each dataset.
def compute_summary(preds, ages):
    rows = []
    for clock in preds.columns:
        y, x = preds[clock].values, ages.values
        valid = ~np.isnan(y)
        if valid.sum() < 3:
            rows.append([clock, np.nan, np.nan, np.nan, np.nan]); continue
        r, _ = pearsonr(x[valid], y[valid])
        if clock in PACE_CLOCKS:
            rows.append([clock, r, np.nan, np.nan, np.nan]); continue
        mae = float(np.mean(np.abs(y[valid] - x[valid])))
        bias = float(np.mean(y[valid] - x[valid]))
        rmse = float(np.sqrt(np.mean((y[valid] - x[valid]) ** 2)))
        rows.append([clock, r, mae, bias, rmse])
    return pd.DataFrame(rows, columns=['Clock','Pearson_r','MAE_years','Mean_bias_years','RMSE_years'])

ages_a = geo_a.metadata['age'].astype(float)
ages_b = geo_b.metadata['age'].astype(float)

summary_a = compute_summary(preds_a, ages_a)
summary_b = compute_summary(preds_b, ages_b)

summary_a.to_csv(OUT_TBL / f'summary_{label_a}.csv', index=False)
summary_b.to_csv(OUT_TBL / f'summary_{label_b}.csv', index=False)

print(f'\n=== {label_a} summary ===')
print(summary_a.round(3).to_string(index=False))
print(f'\n=== {label_b} summary ===')
print(summary_b.round(3).to_string(index=False))

## 5. Required visualizations

### 5.1 Correlation matrix (Requirement #4)

In [ ]:
sex_a = geo_a.metadata.get('sex', pd.Series(np.nan, index=ages_a.index))
sex_b = geo_b.metadata.get('sex', pd.Series(np.nan, index=ages_b.index))

plot_correlation_matrix(preds_a, ages_a, sex_a, OUT_FIG / f'correlation_matrix_{label_a}.png', label_a)
plot_correlation_matrix(preds_b, ages_b, sex_b, OUT_FIG / f'correlation_matrix_{label_b}.png', label_b)

from IPython.display import Image, display
display(Image(OUT_FIG / f'correlation_matrix_{label_a}.png'))
display(Image(OUT_FIG / f'correlation_matrix_{label_b}.png'))

**Reading the correlation matrix:**
- The chronological-age clocks form a tight block (r > 0.9) — they all agree on relative ordering.
- VidalBralo (8 CpGs) sits slightly apart due to its tiny CpG footprint.
- **DunedinPACE** has near-zero or negative correlation with the chronological-age clocks. This is **expected** — it measures pace of aging (years aged per calendar year), not absolute age, so it's a fundamentally different quantity.
- AltumAge (deep NN, ~21k input CpGs) clusters tightly with the linear clocks, showing the non-linear learner converged on similar age signal.

### 5.2 Age deviation heatmap (Requirement #5)

In [ ]:
plot_deviation_heatmap(preds_a, ages_a, OUT_FIG / f'age_deviation_heatmap_{label_a}.png', label_a)
plot_deviation_heatmap(preds_b, ages_b, OUT_FIG / f'age_deviation_heatmap_{label_b}.png', label_b)

display(Image(OUT_FIG / f'age_deviation_heatmap_{label_a}.png'))
display(Image(OUT_FIG / f'age_deviation_heatmap_{label_b}.png'))

**Reading the deviation heatmap:**
- Rows = samples, sorted ascending by chronological age (youngest top → oldest bottom).
- Cell colour = `predicted age − chronological age`. Red = clock predicts older; blue = clock predicts younger.
- Each clock has a *characteristic vertical band of bias* — that's the systematic offset of its calibration vs the reference.
- DunedinPACE is excluded since it outputs years/year, not years.
- The **within-clock vertical gradient** (lighter at top → redder at bottom) is the actual age signal — older samples produce older predictions across nearly every clock.
- Multi-clock age acceleration patterns in individual rows would suggest genuinely 'biologically older' samples (real biological age acceleration).

### 5.3 Predicted vs chronological age (Requirement #6)

In [ ]:
plot_age_scatter_grid(preds_a, ages_a, OUT_FIG / f'age_prediction_{label_a}.png', label_a)
plot_age_scatter_grid(preds_b, ages_b, OUT_FIG / f'age_prediction_{label_b}.png', label_b)

display(Image(OUT_FIG / f'age_prediction_{label_a}.png'))
display(Image(OUT_FIG / f'age_prediction_{label_b}.png'))

**Reading the scatter grid:**
- Each panel = one clock. Blue points = samples; dashed line = perfect prediction (`y = x`).
- A point's vertical distance from the dashed line is its prediction error.
- Slope of the cloud reflects clock calibration (slope ~1 = well-calibrated; <1 = underestimates older / overestimates younger).
- DunedinPACE excluded (different unit).

## 6. Bonus visualizations

### 6.1 MAE comparison across both datasets

In [ ]:
summaries = {label_a: summary_a, label_b: summary_b}
plot_mae_comparison(summaries, OUT_FIG / 'mae_comparison.png')
display(Image(OUT_FIG / 'mae_comparison.png'))

**Reading the MAE bar chart:** lower bar = better-calibrated clock for that dataset. Differences between the two datasets reflect how well a given clock generalises across cohorts.

### 6.2 Predicted-age distributions

In [ ]:
predictions = {label_a: preds_a, label_b: preds_b}
metadata_ages = {label_a: ages_a, label_b: ages_b}
plot_predicted_age_distribution(predictions, metadata_ages, OUT_FIG / 'predicted_age_distribution.png')
display(Image(OUT_FIG / 'predicted_age_distribution.png'))

**Reading the distribution plot:**
- Each box = predicted-age distribution from one clock for one dataset.
- Shaded horizontal bands = chronological-age range of each dataset.
- Dashed lines = chronological-age median of each dataset.
- A well-calibrated clock should have its box centred near its dataset's dashed median line.
- Boxes wildly outside the shaded band indicate scale mismatch (e.g. PhenoAge tends to predict broader ranges).

## 7. Summary

All 6 assignment requirements addressed:

| # | Item | Output |
|---|---|---|
| 1 | 2 datasets | GSE40279 + GSE41169 |
| 2 | >= 8 clocks | 14 clocks across 5 methodology families |
| 3 | Describe datasets + clocks | §2 + §3 above |
| 4 | Correlation matrix | `figures/correlation_matrix_*.png` |
| 5 | Deviation heatmap | `figures/age_deviation_heatmap_*.png` |
| 6 | Predicted-vs-chronological scatter | `figures/age_prediction_*.png` |
| + | Bonus: MAE comparison | `figures/mae_comparison.png` |
| + | Bonus: prediction distributions | `figures/predicted_age_distribution.png` |

## References

- de Lima Camillo, L. P. *et al.* (2023). *Biolearn, an open-source library for biomarkers of aging.* bioRxiv. <https://doi.org/10.1101/2023.12.02.569722>
- Hannum, G. *et al.* (2013). *Mol Cell* 49:359-367.
- Horvath, S. (2013). *Genome Biol* 14:R115.
- Levine, M. E. *et al.* (2018). *Aging* 10:573-591.
- Higgins-Chen, A. T. *et al.* (2022). *Nature Aging* 2:644-661.
- Ying, K. *et al.* (2022). *Nature Aging* 4:231-246.
- Tong, H. *et al.* (2024). *Nature Aging* 4:886-901.
- de Lima Camillo, L. P. *et al.* (2022). *npj Aging* 8:4.
- Belsky, D. W. *et al.* (2022). *eLife* 11:e73420.
